In [13]:
!pip install python-dotenv

In [14]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

True

In [15]:
usuario_minio = os.getenv("MINIO_ACCESS_KEY")
senha_minio = os.getenv("MINIO_SECRET_KEY")

In [16]:
spark = (
    SparkSession.builder
        .appName("teste-minio")
        .master("local[*]")
        #.master("spark://spark-master:7077")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")

        #config s3a --> minio
        .config("spark.hadoop.fs.s3a.endpoint", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key",usuario_minio)
        .config("spark.hadoop.fs.s3a.secret.key", senha_minio)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

        # Delta lake - obrigatório para usar format("delta")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
)

26/07/16 17:30:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [17]:
df = spark.read.csv("s3a://dados/E-Commerce/Categoria.csv", header=True, inferSchema=True)

In [18]:
df.show()

+---+--------------------+
| id|                name|
+---+--------------------+
|  0|   Moda e Acessórios|
|  1|Cosméticos e Perf...|
|  2|    Eletrodomésticos|
|  3|              Livros|
|  4|           Celulares|
|  5|         Informática|
|  6|    Casa e Decoração|
|  7|         Eletrônicos|
|  8|     Esporte e Lazer|
|  9|  Brinquedos e Games|
+---+--------------------+



In [19]:
(df.write
    .format("delta")
    .mode("overwrite")
    .save("s3a://dados/bronze/Categoria.delta")

)

In [9]:
spark.sql("""
    DESCRIBE DETAIL
    delta.`s3a://dados/bronze/Categoria.delta`
""").show(truncate=False)

+------+------------------------------------+----+-----------+----------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name|description|location                          |createdAt              |lastModified       |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+----+-----------+----------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |f4d76a51-7f06-4a12-a821-695c32bfdde4|NULL|NULL       |s3a://dados/bronze/Categoria.delta|2026-07-14 17:37:37.364|2026-07-16 14:54:45|[]              |1       |958        |{}        |1               |2               |[appendOnly, invariants]|


In [10]:
spark.sql("""
    DESCRIBE extended
    delta.`s3a://dados/bronze/Categoria.delta`
""")

DataFrame[col_name: string, data_type: string, comment: string]

In [11]:
spark.sql("""
    DESCRIBE history
    delta.`s3a://dados/bronze/Categoria.delta`
""").show(truncate=False)

+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|2      |2026-07-16 14:54:45|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |1          |Serializable  |false        |{numFiles -> 1, numOutputRows -> 1

In [12]:
spark.stop()